# Homework 02: SQL Mini-Project with Sakila
## Joins, Aggregation, CTEs, Window Functions, and Performance

**Due date:** _April 3, 2026 by 11:59 PM_

**Points:** 100 total

> **Name (FIRST and LAST)** — Brendan Nemeth

- - -

## Overview and Instructions

**Context:** This homework builds directly on our relational database + SQL module (Days 11–19). You will use a Dockerized MySQL database (the Sakila sample database you set up previously) and a Jupyter notebook to carry out a small, portfolio-ready SQL analysis.

You will:
- Connect to a Dockerized MySQL database from this notebook.
- Design and implement several non-trivial SQL queries using Sakila.
- Use joins, grouped aggregation with `HAVING`, CTEs, window functions, and a brief performance check.
- Practice explicit validation habits (row counts, sanity checks, cross-checks).
- Produce at least one manager-style report table that could stand alone in a portfolio.
- Reflect on your SQL reasoning and debugging process.
- Log any use of AI tools in an **AI Audit Log** section.
- Create a GitHub repository to share your work.

**If you get stuck, have questions, or run into any issues as you work through this assignment**, remember to consult the PCAs, ICAs, slides from class for the **Day 11-19 class periods** -- the slides in particular often have worked examples of queries that are similar to what you'll need to write for this assignment. You are also encouraged to **come to office hours** to get direct support from your instructors! If you can't find out office hours information or need to schedule by appoint, **send us an email or catch one of us at the end of class**.

**Submission:**
- Push this completed notebook (and any small helper files) to a new private GitHub repo called `cmse492-hw02-yourMSUNetID` (make sure you use your actual MSU NetID!). **Details for this process are include in Part 10 below.**
- Submit the GitHub URL via the provided Microsoft form.
- Upload the executed notebook (with outputs) to D2L as a backup.

This notebook is designed to be a **self-contained artifact**: someone with access to your Docker image and credentials should be able to rerun your analysis and understand what you did, why you did it, and how you validated it.

**Grading breakdown:** See section headings for point values.

<div class="alert alert-attention">

**Note about AI tools:** As discussed at the beginning of the course, you are allowed to use AI tools (e.g., ChatGPT, Copilot) in responsible, transparent, and ethical ways. For this particular assignment, if you end up using AI tools for assignment, but you will be expected to document your usage in an "AI Audit Log" section near the end of this notebook. See the details in the "AI Audit Log" section below.

</div>


- - -

## 0. Setup and Database Connection

In this section you will:
- Confirm your Dockerized MySQL + Sakila setup is working.
- Establish a connection from Python to the database using `sqlalchemy`.
- Define a small helper function to run SQL and get results as a `pandas` DataFrame (make sure you look at this function so that you understand how it works and can use it in the assignment!)

**Even though you are being provided with the code cell necessary to connect to the database, you should make sure you carefully review the code and understand how it is working.** If you end up needing to connect to a SQL database on your own in the future, you should know how to do so.

**Remember**: Before you try to connect to the database, make sure your Docker container is running and that the Sakila database is properly set up inside it. If you don't recall how to do this, review the instructions from the Day 17 content and get that set up first.

If you run into an issue with this part of the assignment, contact your instructors _as soon as possible_ so we can help you get it resolved. You will not be able to complete the rest of the assignment without a working database connection, so this is a critical first step.


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Credentials for the local Docker MySQL container (must match docker-compose.yml)
uname = 'sakila'
pwd = 'p_ssW0rd'
hname = 'localhost'
dbname = 'sakila'

engine = create_engine(f'mysql+pymysql://{uname}:{pwd}@{hname}/{dbname}')
connection = engine.connect()

def run_sql(query, params=None):
    """Run a SQL query and return a pandas DataFrame.
    `query` can be a string; `params` is an optional dict for parameterized queries.
    """
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df

# Quick test to confirm that you can connect and query the database
test_df = run_sql("SELECT * FROM film LIMIT 5;")
test_df.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


- - -

## 1. Framing Your Mini-Project (5 points)

Your goal is to design a **small analytic story** using the Sakila database. Think of a manager at a video rental company who wants to understand something about customers, films, inventory, stores, or revenue.

In this section, you will define:
- A brief business/analytic question or set of questions you want to answer using SQL and the audience you are imagining.
- The main tables you expect to use.
- The kinds of outputs you want (e.g., a ranked table, grouped summary, percent-of-total view).

**Note**: you might find that your question and plans evolve as you start writing SQL and seeing results. This is totally normal! Just make sure to document your initial thinking here, and then as you evolve your question and plans, make sure to update this section to reflect that evolution. Make sure to keep your initial question and plans in there as well, so that we can see how your thinking evolved.

**Another note**: You might need to spend some time exploring the Sakila schema and data to come up with a question that is interesting and feasible. You can access the DB using Adminer (via `http://localhost:8080`) to explore the schema and/or use SQL queries to poke at the data as part of your process of coming up with a good question.

If you're really struggling to come up with your "story", you can start working through the SQL exercises in the sections that follow, which require you to come up with individual questions to answer with SQL. **You might find that as you do those exercises, you see a collection of insights that your can develop into your project story.** "Reverse-engineering" your project story from the SQL exercises is a totally valid way to come up with a project question and plan!

### 1.1 Project question and plan

**Prompt:** In 1–2 paragraphs, describe your mini-project:
- What is the main question or set of questions you want to answer using SQL?
- Who is the imagined audience (e.g., store manager, marketing analyst, operations lead)?
- Which core tables do you expect to rely on (list at least 3 Sakila tables and why)?
- What kind of final table(s) or figure(s) do you expect to produce (e.g., top categories by revenue, customer segments by activity)?


> **_The central question of this analysis is whether customers from different countries show measurably different preferences for film genres when renting. More specifically, we want to identify which categories like Action, Comedy, Drama, or Horror are most popular in each country, and whether certain countries rent some genres at a higher or lower rate than the global average. A secondary question is whether some countries show a highly concentrated genre preference versus a more diverse rental pattern. The imagined audience is a marketing analyst looking to do a regional promotion with content made based on a country's preferences. The core tables I will need for this task are rental, customer_list, inventory, film, film_category, category. Using joins, I will be able to find the customer's country and the genre for every movie that is rented. The expected final outputs are a ranked summary table of rental counts by country and genre, and a percent-of-total table showing each genre's share of rentals within each country, making cross-country comparisons intuitive even when raw rental volumes differ significantly between countries._**
>

---

## 2. Joins and Multi-Table Queries (15 points)

In this section, you will write **at least two non-trivial join queries** that connect 2–3 tables with meaningful conditions (not just a single key lookup). For each query:
- Clearly state the question it answers.
- Sketch what a single row in the result represents.
- Write and run the SQL.
- Perform at least one validation check (row counts, spot-check rows against base tables, etc.).


### 2.1 Join Query 1

**Example idea (you may adapt this or pick a different one, do not use exactly this example):**
- "For each rental in a specified 60-day period, show customer name, film title, rental date, and store city."

Based on your question and plans from the previous section, write your own question and design for this first join query. You can use the example above as a template or inspiration, but make sure to write your own unique question and design that fits with your overall mini-project. For this first join query, you're expected to clearly define the following:

1. **Question in words**: Describe what you are trying to learn in one or two sentences.
2. **Row meaning sketch**: What does each row represent? Which columns should appear?
3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?
4. **SQL query**: Write the join with clear aliases.
5. **Validation**: Perform at least one validation check, such as:
   - Compare a row count to a simpler query.
   - Spot-check an individual record across base tables.


> **2.1.a Question and design**
> 
> Question in words: _For each rental transaction, what is the customer's name, their country, the film they rented, and what genre that film belongs to?_
>
> Row meaning sketch: _Each row represents one rental event. The columns should be: rental_id, customer_name, country, film_title, category_name, and rental_date._
>
>Tables and keys: _rental joined to customer_list on rental.customer_id = customer_list.ID
rental joined to inventory on rental.inventory_id = inventory.inventory_id
inventory joined to film on inventory.film_id = film.film_id
film joined to film_category on film.film_id = film_category.film_id
film_category joined to category on film_category.category_id = category.category_id_


In [2]:
# 2.1.b Join Query 1 - SQL

# Now that you've defined your question and design for the first join query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, and to include any necessary WHERE conditions to filter the data as
# needed for your question.

join_query_1 = """
SELECT
r.rental_id,
cl.name AS customer_name,
cl.country,
f.title AS film_title,
c.name AS category_name,
r.rental_date
FROM rental r
JOIN customer_list cl ON r.customer_id = cl.ID
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
ORDER BY r.rental_date DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_1 = run_sql(join_query_1)
df_join_1.head()

,rental_id,customer_name,country,film_title,category_name,rental_date
0,11739,LOUIS LEONE,Philippines,ZHIVAGO CORE,Horror,2006-02-14 15:16:03
1,14616,NEIL RENNER,Vietnam,WORLD LEATHERNECKS,Horror,2006-02-14 15:16:03
2,11676,NATALIE MEYER,Brazil,WOMEN DORADO,Action,2006-02-14 15:16:03
3,15966,JEREMY HURTADO,Brazil,WINDOW SIDE,Travel,2006-02-14 15:16:03
4,13486,NAOMI JENNINGS,India,WILD APOLLO,New,2006-02-14 15:16:03


**2.1.c Validation for Join Query 1**

Describe at least one validation check you ran and what you concluded based on it.

You might:
- Show a supporting SQL snippet in a small code cell (e.g., `COUNT(*)` comparison).
- Spot-check one ID across multiple tables and describe what you saw.


In [12]:
validation_query_1 = """
-- Spot check rental_id = 1
SELECT *
FROM rental r
JOIN customer_list cl ON r.customer_id  = cl.ID
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
WHERE r.rental_id = 1;
"""
run_sql(validation_query_1)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update,ID,name,address,...,replacement_cost,rating,special_features,last_update,film_id,category_id,last_update,category_id,name,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53,130,CHARLOTTE HUNTER,758 Junan Lane,...,21.99,G,Trailers,2006-02-15 05:03:42,80,8,2006-02-15 05:07:09,8,Family,2006-02-15 04:46:27


> **_Using my validation check, you can get one specific row and then manually confirm the film title and category match what film and category show for those IDs directly._**
> 

### 2.2 Join Query 2

Create a **different** join that still aligns with your project theme.

**Example ideas (again, you should adapt these to your project or create your own):**
- Customer-level summary: each row is one customer with their total rentals, number of distinct categories they rent, and home city.
- Inventory-level view: each row is one film copy with film title, store, and how many times it has been rented.

Repeat the pattern as before, clearly defining:
1. Question in words.
2. Row meaning sketch.
3. Tables and keys.
4. SQL query.
5. Validation check(s).

> **2.2.a Question and design**
> 
> Question in words: _For each customer, how many total rentals have they made, how many distinct genres have they rented from, and what country are they from?_
>
> Row meaning sketch: _Each row represents one customer. The columns should be: customer_id, customer_name, country, total_rentals, and distinct_categories._
>
> Tables and keys: _customer_list joined to rental on customer_list.ID = rental.customer_id
rental joined to inventory on rental.inventory_id = inventory.inventory_id
inventory joined to film_category on inventory.film_id = film_category.film_id_

In [3]:
# 2.2.b Join Query 2 - SQL
join_query_2 = """
SELECT
cl.ID AS customer_id,
cl.name AS customer_name,
cl.country,
COUNT(r.rental_id) AS total_rentals,
COUNT(DISTINCT fc.category_id) AS distinct_categories
FROM customer_list cl
JOIN rental r ON cl.ID = r.customer_id
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
GROUP BY cl.ID, cl.name, cl.country
ORDER BY total_rentals DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_2 = run_sql(join_query_2)
df_join_2.head()

,customer_id,customer_name,country,total_rentals,distinct_categories
0,148,ELEANOR HUNT,Runion,46,14
1,526,KARL SEAL,United States,45,15
2,144,CLARA SHAW,Belarus,42,16
3,236,MARCIA DEAN,Philippines,42,15
4,75,TAMMY SANDERS,Taiwan,41,16


**2.2.c Validation for Join Query 2**

As before, describe your validation steps and conclusions for this second join query. You can use similar validation techniques as before or come up with new ones that fit the specific question and data you are working with.

In [10]:
# As appropriate, define helper validation query(ies) for Join Query 2
validation_query_2 = """
# -- Should equal SELECT COUNT(*) FROM rental
SELECT SUM(total_rentals)
FROM (
SELECT COUNT(r.rental_id) AS total_rentals
FROM customer_list cl
JOIN rental r ON cl.ID = r.customer_id
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
GROUP BY cl.ID
) sub;
"""
run_sql(validation_query_2)

,SUM(total_rentals)
0,16044


> **_This validation check is used so that you can see if the number of rentals matches up with the one from the original table. Mine does, so I can continue with my analysis._**

---

## 3. Grouped Aggregation with HAVING (15 points)

In this section you will:
- Write at least one grouped aggregation that uses `GROUP BY` and `HAVING`.
- Apply sanity checks (e.g., sum of group counts vs. raw counts, spot-check of a group).
- Tie the result to a question your audience might actually care about.


### 3.1 Aggregation Query with HAVING (required)

**Example ideas (adapt or design your own to fit your audience and align with your mini-project):**
- "Find film categories with at least 1000 total rentals, ordered from most to least rented."
- "Find customers who have spent at least \$X in total payments."

For your query, **document**:
1. Business question in words (1–2 sentences).
2. Row meaning and expected columns in the grouped result.
3. SQL query with `GROUP BY` and `HAVING`.
4. At least two sanity checks (e.g., compare grouped count sums to raw counts, spot-check a single group).


> **3.1.a Question and design**
> 
> Business question in words: _Which country and genre combinations have generated at least 50 rentals?_
>
> Row meaning sketch: _Each row represents one country/genre pair. The columns should be: country, category_name, and rental_count._

In [ ]:
# 3.1.b Aggregation with GROUP BY + HAVING

# Now that you've defined your question and design for the aggregation query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, to include the appropriate GROUP BY and HAVING clauses, and to filter
# the data as needed for your question.

agg_query = """
SELECT
cl.country,
c.name AS category_name,
COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN customer_list cl ON r.customer_id = cl.ID
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
GROUP BY cl.country, c.name
HAVING COUNT(r.rental_id) >= 50
ORDER BY cl.country, rental_count DESC;
"""

# Once you've defined your query, you can uncomment/edit the lines below to run it and see the results.
df_agg = run_sql(agg_query)
display(df_agg) or df_agg.head()

,country,category_name,rental_count
0,Brazil,Sci-Fi,64
1,Brazil,Sports,57
2,Brazil,Animation,55
3,Brazil,Drama,54
4,Brazil,Documentary,52
...,...,...,...
70,United States,Comedy,59
71,United States,Children,56
72,United States,New,55
73,United States,Classics,54


,country,category_name,rental_count
0,Brazil,Sci-Fi,64
1,Brazil,Sports,57
2,Brazil,Animation,55
3,Brazil,Drama,54
4,Brazil,Documentary,52


**3.1.c Sanity checks for aggregation**

Describe at least **two** checks you performed. Ideas:
- Compare the sum of group counts to a raw `COUNT(*)` over a comparable subset.
- Focus on one group (e.g., one category or one customer), query its underlying rows, and verify the count.
- If applicable, use a `LEFT JOIN` to see categories with zero values and explain what you observe.


In [18]:
# As appropriate, write helper sanity-check queries for aggregation here and
# provide explanation in the next section.

check_query_1 = """
-- Total rentals across all country/genre pairs with >= 50 rentals
SELECT SUM(rental_count)
FROM (
    SELECT COUNT(r.rental_id) AS rental_count
    FROM rental r
    JOIN customer_list cl  ON r.customer_id  = cl.ID
    JOIN inventory i       ON r.inventory_id = i.inventory_id
    JOIN film_category fc  ON i.film_id      = fc.film_id
    JOIN category c        ON fc.category_id = c.category_id
    GROUP BY cl.country, c.name
    HAVING COUNT(r.rental_id) >= 50
) sub;
"""
run_sql(check_query_1)

check_query_2 = """
-- Verify one specific country/genre pair manually
SELECT COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN customer_list cl  ON r.customer_id  = cl.ID
JOIN inventory i       ON r.inventory_id = i.inventory_id
JOIN film_category fc  ON i.film_id      = fc.film_id
JOIN category c        ON fc.category_id = c.category_id
WHERE cl.country = 'India'
  AND c.name = 'Sports';
  """
run_sql(check_query_2)

,rental_count
0,115


> **Sanity Check 1: _Compare grouped sum to raw count: The sum of rental_count across all groups in the HAVING-filtered result will be less than or equal to SELECT COUNT(*) FROM rental, since we are dropping low-activity pairs. Confirm the filtered sum is a reasonable fraction of the total._
>
> Sanity Check 2: _Spot check one group: Pick a specific country and genre from the results and manually count rentals for that combination without the HAVING clause to confirm the number matches._**

- - - 

## 4. Multi-Step Query Using a CTE (15 points)

Now you will use a **CTE (`WITH` clause)** to break a complex analysis into readable steps. You may reuse logic from earlier sections, but the CTE should add structure (e.g., pre-filtering, intermediate grouping) rather than just rename an existing query.

**Example patterns:**
- First CTE: compute customer-level rental counts and total payments.
- Second step: filter to active customers, compute ranks or thresholds on top.

Your CTE query must:
- Use at least one `WITH` clause.
- Do something non-trivial in the CTE (filtering, joining, grouping, etc.).
- Be accompanied by validation (e.g., run parts of the CTE separately, check a subset).

### 4.1 CTE design and explanation

Describe your CTE in words:
- What does the CTE compute?
- What does the outer query do on top of it?
- How does this decomposition make the logic clearer than a single big query would be?


> **_The CTE computes the total number of rentals for every country and genre combination, which is the same grouping logic we built toward in earlier sections. Doing this in a CTE lets us treat that grouped result as a clean named table in the next step, rather than nesting a subquery inside the outer query.
The outer query then calculates each genre's share of rentals within its country as a percentage, using a window function (SUM() OVER) to get the country-level total without collapsing the rows. This gives us a percent-of-total view per country, which is the core analytic output described in section 1.
Without the CTE, computing a percent-of-total would require either a self-join or a deeply nested subquery that mixes raw joins, grouping, and window logic all at once. Breaking it into two steps makes each part easier to read, test, and debug independently._**

### 4.2 Writing the CTE query

Now that you have a plan for your CTE, write the SQL query that implements it.

In [ ]:
# 4.2 CTE-based query

# Now that you've designed your CTE, write the SQL query that implements it.
# Make sure to use clear aliases for tables and columns, and to structure the
# CTE in a way that breaks down the logic into understandable steps.

cte_query = """
WITH country_genre_counts AS (
SELECT
cl.country,
c.name AS category_name,
COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN customer_list cl ON r.customer_id = cl.ID
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
GROUP BY cl.country, c.name
)
SELECT
country,
category_name,
rental_count,
SUM(rental_count) OVER (PARTITION BY country) AS country_total,
ROUND(
100.0 * rental_count /
SUM(rental_count) OVER (PARTITION BY country), 2
) AS pct_of_country
FROM country_genre_counts
ORDER BY country, pct_of_country DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_cte = run_sql(cte_query)
display(df_cte) or df_cte.head()

,country,category_name,rental_count,country_total,pct_of_country
0,Afghanistan,Comedy,3,18,16.67
1,Afghanistan,New,2,18,11.11
2,Afghanistan,Drama,2,18,11.11
3,Afghanistan,Documentary,2,18,11.11
4,Afghanistan,Classics,2,18,11.11
...,...,...,...,...,...
1581,Zambia,Family,2,33,6.06
1582,Zambia,Sports,1,33,3.03
1583,Zambia,Children,1,33,3.03
1584,Zambia,Travel,1,33,3.03


,country,category_name,rental_count,country_total,pct_of_country
0,Afghanistan,Comedy,3,18,16.67
1,Afghanistan,New,2,18,11.11
2,Afghanistan,Drama,2,18,11.11
3,Afghanistan,Documentary,2,18,11.11
4,Afghanistan,Classics,2,18,11.11


### 4.3 CTE validation

Run at least one validation step focused on the **intermediate CTE logic**, such as:
- Select a subset of the CTE output (e.g., `LIMIT 10`) and manually compare to base tables.
- Recompute a simpler version of a metric (e.g., total payments) for one customer and confirm it matches the CTE result.

In [19]:
# As appropriate, write helper queries to inspect the CTE result or base tables
check_cte_row = """
-- Run just the CTE body to verify it
SELECT
    cl.country,
    c.name             AS category_name,
    COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN customer_list cl  ON r.customer_id  = cl.ID
JOIN inventory i       ON r.inventory_id = i.inventory_id
JOIN film_category fc  ON i.film_id      = fc.film_id
JOIN category c        ON fc.category_id = c.category_id
GROUP BY cl.country, c.name
ORDER BY cl.country, rental_count DESC;
"""
run_sql(check_cte_row)

,country,category_name,rental_count
0,Afghanistan,Comedy,3
1,Afghanistan,Classics,2
2,Afghanistan,New,2
3,Afghanistan,Drama,2
4,Afghanistan,Documentary,2
...,...,...,...
1581,Zambia,Horror,2
1582,Zambia,Children,1
1583,Zambia,Sports,1
1584,Zambia,Sci-Fi,1


> **_Copy just the inner SELECT and run it on its own to confirm the row counts and rental totals look reasonable before the outer query touches it._**

---

## 5. Window Function with `OVER (...)` (15 points)

Next, you will write at least one query that uses a **window function**. Options include:
- `ROW_NUMBER()`, `RANK()` or `DENSE_RANK()` within a category (e.g., top customers per store).
- A percent-of-total calculation using `SUM(...) OVER ()` or `SUM(...) OVER (PARTITION BY ...)`.

Your window query should:
- Use `OVER (...)` in a meaningful way.
- Produce a result that could help your audience understand ranking or relative importance.
- Include at least one validation step (e.g., verify that percentages sum to ~100%).


### 5.1 Window function design

Describe your plan:
- What are you ranking or computing percentages over?
- What does each row represent in the final result?
- How will your audience interpret the window column(s)?


> **_Within each country, we are ranking genres by their total rental count from highest to lowest. This tells the marketing analyst which genre is #1, #2, #3, etc. in each country without having to manually scan through a long sorted table. Each row is still one country/genre pair, but now it also carries a genre_rank column showing where that genre lands within its country. A rank of 1 means that genre is the most rented in that country. The analyst can quickly filter to genre_rank = 1 to see each country's single top genre, or filter to genre_rank <= 3 to see the top 3 genres per country for a regional campaign shortlist._**

### 5.2 Writing the window function query

Now that you have a plan for your window function, write the SQL query that implements it.

In [6]:
# 5.2 Window function query

# Now that you've designed your window function query, write the SQL to
# implement it. Make sure to use clear aliases for tables and columns, to
# structure the query in a way that breaks down the logic into understandable
# steps, and to include the appropriate window function(s) with meaningful
# OVER(...) clauses.

window_query = """
WITH country_genre_counts AS (
    SELECT
        cl.country,
        c.name              AS category_name,
        COUNT(r.rental_id)  AS rental_count
    FROM rental r
    JOIN customer_list cl  ON r.customer_id  = cl.ID
    JOIN inventory i       ON r.inventory_id = i.inventory_id
    JOIN film_category fc  ON i.film_id      = fc.film_id
    JOIN category c        ON fc.category_id = c.category_id
    GROUP BY cl.country, c.name
)
SELECT
    country,
    category_name,
    rental_count,
    DENSE_RANK() OVER (
        PARTITION BY country
        ORDER BY rental_count DESC
    ) AS genre_rank
FROM country_genre_counts
ORDER BY country, genre_rank;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_window = run_sql(window_query)
display(df_window) or df_window.head()

,country,category_name,rental_count,genre_rank
0,Afghanistan,Comedy,3,1
1,Afghanistan,Classics,2,2
2,Afghanistan,Documentary,2,2
3,Afghanistan,New,2,2
4,Afghanistan,Drama,2,2
...,...,...,...,...
1581,Zambia,Horror,2,3
1582,Zambia,Children,1,4
1583,Zambia,Sports,1,4
1584,Zambia,Sci-Fi,1,4


,country,category_name,rental_count,genre_rank
0,Afghanistan,Comedy,3,1
1,Afghanistan,Classics,2,2
2,Afghanistan,Documentary,2,2
3,Afghanistan,New,2,2
4,Afghanistan,Drama,2,2


### 5.3 Window function validation

Identify at least one validation you can perform, such as:
- Summing `pct_of_total` and confirming it is close to 100 (allowing for rounding).
- Checking that the ranking order matches the sorted totals.
- Comparing one row’s percent-of-total to a hand calculation.

In [20]:
# As appropriate, write helper queries or pandas checks for window function results
# e.g., df_window['pct_of_total'].sum()
window_validation = '''
-- Each country should appear exactly once
SELECT *
FROM (
    WITH country_genre_counts AS (
        SELECT
            cl.country,
            c.name              AS category_name,
            COUNT(r.rental_id)  AS rental_count
        FROM rental r
        JOIN customer_list cl  ON r.customer_id  = cl.ID
        JOIN inventory i       ON r.inventory_id = i.inventory_id
        JOIN film_category fc  ON i.film_id      = fc.film_id
        JOIN category c        ON fc.category_id = c.category_id
        GROUP BY cl.country, c.name
    )
    SELECT
        country,
        category_name,
        rental_count,
        DENSE_RANK() OVER (
            PARTITION BY country ORDER BY rental_count DESC
        ) AS genre_rank
    FROM country_genre_counts
) ranked
WHERE genre_rank = 1
ORDER BY country;
'''
window_val_df = run_sql(window_validation)
display(window_val_df)

,country,category_name,rental_count,genre_rank
0,Afghanistan,Comedy,3,1
1,Algeria,Sci-Fi,12,1
2,American Samoa,Sports,5,1
3,Angola,Animation,7,1
4,Angola,Comedy,7,1
...,...,...,...,...
143,"Virgin Islands, U.S.",Animation,7,1
144,Yemen,Action,15,1
145,Yugoslavia,Animation,7,1
146,Zambia,Games,4,1


> **_Pull only the top-ranked genre per country and confirm each country appears exactly once, and that the genre shown has the highest rental count for that country. The only exception is if multiplw genres are tied for first for a country._**

---

## 6. Performance Check with `EXPLAIN` and/or Index (5 points)

Pick **one** of your more complex queries from the earlier sections (join, aggregation, or window-based) and perform a brief performance analysis using MySQL's `EXPLAIN` statement.

You should:
- State which query you are examining and what aspect of its execution plan you want to inspect.
- Run `EXPLAIN` on the query and capture the output in a DataFrame.
- Identify at least one aspect of the plan (e.g., table scan vs. index usage, join order, estimated rows) and explain why it seems reasonable or concerning.
- Optionally, propose or implement a simple index and compare the `EXPLAIN` output before vs. after.

You do **not** need to time queries precisely or chase microseconds. The focus is on reading and interpreting the query plan.

**Note**: You might not see any "red flags" in the `EXPLAIN` output, especially on a small dataset like Sakila. That's totally fine! The point is to practice reading the plan and thinking about what it means.

### 6.1 Choose a query and plan your analysis

Before running `EXPLAIN`, take a moment to describe your plan:
1. Which query from an earlier section are you examining (e.g., "Section 3 aggregation by category", "Section 5 window query")?
2. Why did you choose this query (e.g., it joins many tables, it aggregates a large dataset)?
3. What do you expect to see in the `EXPLAIN` output (e.g., index usage on join keys, a full scan on a particular table)?

> **_Write your plan here._**
>

### 6.2 Run EXPLAIN on your chosen query

Now that you have a plan, run `EXPLAIN` on the query you chose. Note that you can prefix any SQL query with `EXPLAIN` to get the execution plan — `run_sql()` works with `EXPLAIN` statements just like any other query.

In [ ]:
# 6.2 Run EXPLAIN on your chosen query

# Prefix your query with EXPLAIN to capture the execution plan.
# Note: run_sql() works with EXPLAIN statements just like any other query.

explain_query = """
-- TODO: Replace with EXPLAIN for one of your actual queries from earlier sections.
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
# df_explain = run_sql(explain_query)
# display(df_explain) or df_explain.head()

### 6.3 Interpret the EXPLAIN output

Now that you have the `EXPLAIN` output, interpret it in a few sentences. Address at least one of the following:
- Is MySQL using an index or doing a full table scan on any of the tables? Does this seem reasonable given the query structure?
- What is the estimated number of rows being scanned for each table (the `rows` column)? Does this seem appropriate?
- Is the join order what you would expect? Why or why not?
- Based on the plan, would you suggest any changes to improve performance (e.g., adding an index, restructuring the query)?

In [ ]:
# Optional: use this cell for any helper queries or pandas checks that
# support your interpretation of the EXPLAIN output.
# For example, you could check the actual row count of a table where EXPLAIN
# shows a large row estimate.

> **_Write your performance interpretation here (2–5 sentences)._**
>

### 6.4 Optional: Propose or add an index

If you'd like to go further and it seems like an optimization is needed, you can propose or create a simple index and compare the `EXPLAIN` output before vs. after. Document:
- Which column(s) you chose and why (e.g., frequently used in `WHERE` or join conditions).
- Whether the `EXPLAIN` output changed in a way that suggests better performance (e.g., less full-table scanning, lower estimated rows).

In [ ]:
# Optional: create an index and re-run EXPLAIN to compare.
# create_index_sql = """
# -- TODO: Replace with your CREATE INDEX statement.
# -- Example: CREATE INDEX idx_rental_date ON rental(rental_date);
# """
# run_sql(create_index_sql)

# explain_after_index = """
# -- TODO: Replace with your EXPLAIN query after adding the index.
# """
# run_sql(explain_after_index)

---

## 7. Manager-Style Report Table (10 points)

Design **one table** that you would feel comfortable showing to your imagined audience or including in a portfolio. This table should:
- Be relatively **wide** (e.g., 4–8 columns) with clear, human-readable column labels.
- Summarize something decision-relevant (e.g., categories by revenue and share, top customers per store).
- Optionally use `CASE` expressions or window functions to create computed or pivot-like columns.

You may reuse or adapt one of your earlier queries if it already reads like a manager report, or you may write a dedicated query here.

### 7.1 Describe your report

Before writing the SQL, describe your report:
1. Who is the audience for this table (e.g., store manager, marketing analyst, operations lead)?
2. What decision or question does this table support?
3. Which columns will appear, and what does each one tell the audience?
4. Will you reuse a query from an earlier section or write a new one?

> **_Audience: A marketing analyst who wants a ready-to-share summary they can bring into a meeting or drop into a slide deck.
>Decision this table supports: Which genres should we prioritize in regional promotions for each country? The table lets the analyst quickly see, for any country, how each genre ranks, how many rentals it drove, what share of that country's total rentals it represents, and whether that genre is performing above or below the global average share for that genre.
>Columns and what each one tells the audience:

>Country: the customer's home country
>Genre: the film category name
>Rentals: raw count of rentals for that country/genre pair
>Country Total Rentals: total rentals across all genres for that country (gives context to the raw count)
>% of Country Rentals: that genre's share of the country's total rentals
>Global Genre %: what share that same genre represents globally across all countries
>vs. Global Avg: a CASE-derived label ("Above Avg", "Below Avg", or "On Par") comparing the country's genre share to the global share, making it easy to spot over- and under-indexing at a glance
>Genre Rank in Country: the genre's rank within that country by rental count

>Will you reuse a query? This builds on and extends the CTE from section 4 and the window function from section 5, adding the global >genre percentage and the CASE label as new computed columns._**
>

### 7.2 Write the report query

Now write the SQL query that produces your report table. Aim for clear, human-readable column aliases and a layout that could stand on its own in a portfolio or presentation.

In [8]:
# 7.2 Report query SQL

# Now that you've described your report, write the SQL that produces it.
# Aim for clear column aliases and a result that could stand on its own in
# a portfolio or presentation.

report_query = """
WITH country_genre_counts AS (
SELECT
cl.country,
c.name AS genre,
COUNT(r.rental_id) AS rentals
FROM rental r
JOIN customer_list cl ON r.customer_id = cl.ID
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
GROUP BY cl.country, c.name
),
global_genre_totals AS (
SELECT
genre,
SUM(rentals) AS global_genre_rentals,
ROUND(100.0 * SUM(rentals) / SUM(SUM(rentals)) OVER (), 2) AS global_genre_pct
FROM country_genre_counts
GROUP BY genre
)
SELECT
cgc.country AS `Country`,
cgc.genre AS `Genre`,
cgc.rentals AS `Rentals`,
SUM(cgc.rentals) OVER (PARTITION BY cgc.country) AS `Country Total Rentals`,
ROUND(100.0 * cgc.rentals / SUM(cgc.rentals) OVER (PARTITION BY cgc.country), 2) AS `% of Country Rentals`,
ggt.global_genre_pct AS `Global Genre %`,
CASE
WHEN ROUND(100.0 * cgc.rentals / SUM(cgc.rentals) OVER (PARTITION BY cgc.country), 2) > ggt.global_genre_pct + 1  THEN 'Above Avg'
WHEN ROUND(100.0 * cgc.rentals / SUM(cgc.rentals) OVER (PARTITION BY cgc.country), 2) < ggt.global_genre_pct - 1  THEN 'Below Avg'
ELSE 'On Par'
END AS `vs. Global Avg`,
DENSE_RANK() OVER (
PARTITION BY cgc.country
ORDER BY cgc.rentals DESC
) AS `Genre Rank in Country`
FROM country_genre_counts cgc
JOIN global_genre_totals ggt ON cgc.genre = ggt.genre
ORDER BY cgc.country, `Genre Rank in Country`;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_report = run_sql(report_query)
display(df_report) or df_report.head()

,Country,Genre,Rentals,Country Total Rentals,% of Country Rentals,Global Genre %,vs. Global Avg,Genre Rank in Country
0,Afghanistan,Comedy,3,18,16.67,5.87,Above Avg,1
1,Afghanistan,New,2,18,11.11,5.86,Above Avg,2
2,Afghanistan,Documentary,2,18,11.11,6.54,Above Avg,2
3,Afghanistan,Classics,2,18,11.11,5.85,Above Avg,2
4,Afghanistan,Drama,2,18,11.11,6.61,Above Avg,2
...,...,...,...,...,...,...,...,...
1581,Zambia,Family,2,33,6.06,6.83,On Par,3
1582,Zambia,Sports,1,33,3.03,7.35,Below Avg,4
1583,Zambia,Children,1,33,3.03,5.89,Below Avg,4
1584,Zambia,Sci-Fi,1,33,3.03,6.86,Below Avg,4


,Country,Genre,Rentals,Country Total Rentals,% of Country Rentals,Global Genre %,vs. Global Avg,Genre Rank in Country
0,Afghanistan,Comedy,3,18,16.67,5.87,Above Avg,1
1,Afghanistan,New,2,18,11.11,5.86,Above Avg,2
2,Afghanistan,Documentary,2,18,11.11,6.54,Above Avg,2
3,Afghanistan,Classics,2,18,11.11,5.85,Above Avg,2
4,Afghanistan,Drama,2,18,11.11,6.61,Above Avg,2


### 7.3 Optional: polish in pandas

If you wish, you can use `pandas` to make the report output more presentation-ready:
- Rename columns for readability.
- Format numeric columns (e.g., round percentages, add currency formatting).
- Select or reorder columns for clarity.

This is optional but can help the output feel more portfolio-ready.

In [ ]:
# Optional: light pandas polishing for the report table, then display it.
# For example, you could format the revenue column as currency, or add a total row at the bottom.

# Put your Pandas commands to polish the report with pandas here (example: format revenue as currency):


# Display the polished report
# display(df_report)

---

## 8. Reflection on SQL Reasoning and Debugging (5 points)

Write **1–2 paragraphs** reflecting on your process. Address at least:
- How you broke down your mini-project into smaller SQL questions, or if your mini-project emerged/evolved as you explored the data and wrote individual queries.
- Any bugs, confusing results, or dead ends you hit and how you debugged them (e.g., row-count checks, `LIMIT`, CTE inspection, switching between Adminer and this notebook).
- Any tradeoffs you noticed between expressing logic with CTEs, `GROUP BY`, and window functions (e.g., readability vs. performance, ease of validation) either for this specific assignment or in general based on your experience.


> **_The project started with a single broad question about whether customers from different countries prefer different genres, and breaking that down into SQL steps made it a lot more manageable. The first join query was really just about confirming the data was connected correctly, getting one flat row per rental with country and genre attached. From there it was natural to move toward aggregation, asking how many rentals each country/genre pair had, and then layering on the percent-of-total calculation to make cross-country comparisons fair. The CTE structure ended up being a natural fit for this because the grouping logic needed to happen before the window functions could run on top of it, so splitting those two steps into a CTE and an outer query made the intent of each part much clearer than one deeply nested query would have.
The trickiest part was making sure the percent-of-total and the global genre average were computed correctly and not accidentally mixing row-level and group-level logic in the same expression. Running the CTE body in isolation first was the most useful debugging habit throughout the project, since it let me verify the grouped counts looked right before trusting the window function results built on top of them. One tradeoff worth noting is that CTEs make queries much easier to read and validate in pieces, but for larger datasets they can sometimes prevent the optimizer from pushing filters down efficiently, which a well-structured subquery might handle better. For a dataset the size of Sakila this was not a concern, but it is something to keep in mind when writing similar queries against production-scale tables._**

---

## 9. AI Audit Log (5 points)

This section is about **transparent, professional use of AI tools** (including systems like ChatGPT, GitHub Copilot, Perplexity, or others). You will not lose points for using AI, but you **must** document how you used it.

Answer the prompts below honestly. If you did **not** use any AI tools, you can say so explicitly.


### 9.1 Tools used

- List any AI tools you used while working on this assignment (e.g., ChatGPT, Copilot, Perplexity, Claude, etc.). If none, write "None".
- For each tool, briefly describe what you asked it to help with (e.g., "helped me remember `CREATE INDEX` syntax", "suggested a pattern for a window function").

> **_I used Claude throughout this assignment to help design and structure my SQL queries. Specifically, I used it to work through the join path logic connecting customer_list, rental, inventory, film_category, and category, and to confirm that the foreign keys between those tables were sufficient to answer my analytic question. I also used it to help think through the structure of the CTE and window function queries, particularly around how to layer the percent-of-total calculation on top of a grouped result and how to incorporate the global genre average comparison in the final manager report table. The SQL logic and overall query designs were developed through that back-and-forth reasoning process rather than copied from external sources._**

### 9.2 Directly used vs. adapted

- Did you paste any AI-generated SQL or code directly into your notebook with minimal changes? If so, point to where (e.g., "Section 5 window query") and **make sure you add a comment in the code cell acknowledging the AI contribution**.
- For code or queries that you **adapted**, briefly describe what you changed and why (e.g., "adjusted table names to match Sakila, simplified to match our validation habits").

> **_Most of the queries in this notebook were used fairly directly from what Claude generated, with minor adaptations to fit the actual column names and aliases I wanted in my output. For example, the CTE query in section 4 and the window function query in section 5 were both generated through the conversation and pasted in with small tweaks like renaming output columns to be more readable and adjusting the ORDER BY to match how I wanted the results sorted. The manager report query in section 7 required the most adaptation since I had to make sure the CASE threshold logic (using plus or minus 1 percentage point as the cutoff for "Above Avg" vs "Below Avg") made sense for my data before committing to it._**

### 9.3 Validation and trust

- How did you validate any AI-suggested queries or patterns before trusting them? (e.g., row-count checks, comparing against a simpler query, sanity checking outputs.)
- Are there any parts of your notebook where you are **not fully confident** in the result? If so, describe them, why you're lacking confidence, and what additional checks you would run if you had more time (e.g., "I wasn't sure if the window function was partitioned correctly, I would want to double-check the logic and maybe test it on a smaller dataset if I had more time").

> **_For every query Claude suggested, I ran the validation checks described in each section, mainly row count comparisons against simpler queries and spot checks on individual countries or customers to make sure the numbers matched what I would expect from a manual count. The area where I feel least fully confident is the global genre percentage calculation in the manager report query. The SUM(SUM(rentals)) OVER () pattern inside a CTE is a bit unusual and while the percentages summed to approximately 100 in my check, I would want to test it on a smaller hand-counted subset to be completely sure the window is scoping correctly across the entire dataset and not accidentally partitioning in a way that is hard to see just from the output. If I had more time I would isolate that CTE, pull the raw numbers for two or three genres, and verify the global percentages by hand before fully trusting that column._**

- - -

## 10. Creating your GitHub repo and pushing your code (10 points)

Now that you've finished your notebook, it's time to create a GitHub repository and push your code. 

**The goal for this section is to build a repository that someone could clone, spin up the Docker container to access the SQL database, and execute your notebook from top to bottom.**

To achieve this, follow these steps:

1. Create a new private repository on GitHub named `cmse492-hw02-yourMSUNetID` (replace `yourMSUNetID` with your actual MSU NetID).
2. Initialize the repository with a README file.
3. Clone the repository to your local machine.
4. Copy your completed notebook into the cloned repository folder. **Commit the fully-executed version so that someone could see the output _without_ having to run it, but using the Docker set up they should be able to execute it, if needed.**
5. Add the necessary Docker compose file needed to run the Sakila database (you can reuse the one provided in the course materials).
6. Update the README so that it provides an overview of the project, what you accomplished and the skills you're showcasing, and instructions on how to run the notebook and connect to the database (make sure you include information for starting up the Docker containers!).

    - You should be able to share this repository with a friend, co-worker, or future employer and they should be able to understand what you did, why you did it, and how to run your notebook and see your analysis in action.
    
7. Use Git commands to add, commit, and push your changes to GitHub. 

- - -
After you're made sure the GitHub repo is set up correctly, **Submit this notebook to D2L** and **Fill out the MS Form below** to finalize your submission.  
(If the form won't render in your notebook for some reason, you can also access it directly here: https://forms.office.com/r/Ppk0tnVkc8)

In [ ]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/Ppk0tnVkc8" 
	width="800" 
	height="800px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

- - -
### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework" section, find the appropriate submission link, and upload it there. **Make sure your instructors have access to your private GitHub repo by adding them as collaborators!**

See you in class!

---
&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.